## XBRL US API - Python example  
This sample Python code queries assertion data for SEC filings.

### Authenticate for access token 
Click the run button and enter your XBRL US Web account email, account password, Client ID, and secret (get these from https://xbrl.us/access-token), pressing the Enter key on the keyboard after each entry.

XBRL US limits records returned for a query to improve efficiency; this script loops to collect all data from the Public Filings Database for a query. **Non-members might not be able to return all data for a query** - join XBRL US for comprehensive access - https://xbrl.us/join.

In [1]:
# @title
import os, re, sys, json
import requests
import pandas as pd
from IPython.display import display, HTML
import numpy as np
import getpass
from datetime import datetime
import urllib
from urllib.parse import urlencode

api = input('Enter subdomain ("api" or left blank to query public results, otherwise enter a value) ') or 'api'
baseurl = 'https://' + api + '.xbrl.us/'

class tokenInfoClass:
    access_token = None
    refresh_token = None
    email = None
    username = None
    client_id = None
    client_secret = None
    authurl = baseurl + 'oauth2/token'
    headers = {"Content-Type": "application/x-www-form-urlencoded"}

def refresh(info):
    refresh_auth = {
                'client_id': info.client_id,
                'client_secret' : info.client_secret,
                'grant_type' : 'refresh_token',
                'platform' : 'ipynb',
                'refresh_token' : info.refresh_token
                }
    refreshres = requests.post(info.authurl, data=refresh_auth, headers=info.headers)
    refresh_json = refreshres.json()
    info.access_token = refresh_json.get('access_token')
    info.refresh_token = refresh_json.get('refresh_token')
    print('Your access token(%s) is refreshed for 60 minutes. If it expires again, run this cell to generate a new token and continue to use the query cells below.' % (info.access_token))
    return info

tokenInfo = tokenInfoClass()

# Helper to prompt only if value is missing
def prompt_if_missing(value, prompt_text, secret=False):
    if value:
        return value
    if secret:
        return getpass.getpass(prompt=prompt_text)
    return input(prompt_text)

# Load creds.json (if exists) and ask for user confirmation
creds = {}
use_creds_file = False
if os.path.exists('creds.json'):
    try:
        with open('creds.json', 'r') as f:
            creds = json.load(f)
        confirm = input("Would you like to use credentials on file for this? (yes/no): ").strip().lower()
        use_creds_file = confirm in ('yes', 'y')
        if use_creds_file:
            print("Using credentials on file")
        else:
            print("Enter credentials manually")
            creds = {}
    except Exception as e:
        print("Warning: failed to read creds.json:", e)
        creds = {}

use_test = 'test' in api.lower()

if creds and use_creds_file:
    # Determine credential source: prod > test > top-level > prompts
    selected = None
    
    # First, try nested objects if they exist (e.g., { "prod": {...}} or { "test": {...}})
    if use_test and isinstance(creds.get('test'), dict):
        selected = creds['test']
    elif not use_test and isinstance(creds.get('prod'), dict):
        selected = creds['prod']
    
    # Next, try prefixed keys (prod*/test* if requested)
    if not selected:
        selected = {}
        keys = ['email', 'password', 'client_id', 'client_secret']
        prefix = 'test' if use_test else 'prod'
        for k in keys:
            prefixed_key = prefix + k
            if creds.get(prefixed_key):
                selected[k] = creds.get(prefixed_key)
            # fall back to top-level key if prefix variant not found
            elif creds.get(k):
                selected[k] = creds.get(k)
    
    # Verify we have all required keys
    if not all(selected.get(k) for k in ('email', 'password', 'client_id', 'client_secret')):
        # Fill in missing values from prompts
        selected = {
            'email': selected.get('email'),
            'password': selected.get('password'),
            'client_id': selected.get('client_id'),
            'client_secret': selected.get('client_secret')
        }
    
    # Assign values, prompting for any missing ones
    tokenInfo.email = prompt_if_missing(selected.get('email'), 'Enter your XBRL US Web account email: ')
    tokenInfo.password = prompt_if_missing(selected.get('password'), 'Password: ', secret=True)
    tokenInfo.client_id = prompt_if_missing(selected.get('client_id'), 'Client ID: ', secret=True)
    tokenInfo.client_secret = prompt_if_missing(selected.get('client_secret'), 'Secret: ', secret=True)

    cred_source = 'test credentials' if use_test else 'prod credentials'
    print(f'Using {cred_source}')
else:
    # No creds.json or user declined — prompt the user
    tokenInfo.email = input('Enter your XBRL US Web account email: ')
    tokenInfo.password = getpass.getpass(prompt='Password: ')
    tokenInfo.client_id = getpass.getpass(prompt='Client ID: ')
    tokenInfo.client_secret = getpass.getpass(prompt='Secret: ')

body_auth = {'username' : tokenInfo.email,
            'client_id': tokenInfo.client_id,
            'client_secret' : tokenInfo.client_secret,
            'password' : tokenInfo.password,
            'grant_type' : 'password',
            'platform' : 'ipynb' }

# Make auth request
payload = urlencode(body_auth)
res = requests.request("POST", tokenInfo.authurl, data=payload, headers=tokenInfo.headers)
auth_json = res.json()

if 'error' in auth_json:
    print("\n\nThere was a problem generating the access token: %s  Run the first cell again and enter the credentials." % (auth_json.get('error_description', auth_json)))
else:
    tokenInfo.access_token = auth_json.get('access_token')
    tokenInfo.refresh_token = auth_json.get('refresh_token')
    if tokenInfo.access_token and tokenInfo.refresh_token:
        print ("\n\nYour access token expires in 60 minutes. After it expires, it should be regenerated automatically.  If not, run the cell rerun the first query cell. \n\nFor now, skip ahead to the section 'Make a Query'.")
    else:
        print("\n\nAuthentication completed but tokens were not returned. Response: {}".format(auth_json))

#print(vars(tokenInfo))
if tokenInfo.access_token and tokenInfo.refresh_token:
    print('\n\naccess token: ' + tokenInfo.access_token + ' refresh token: ' + tokenInfo.refresh_token)
else:
    print('\n\nNo access token was generated. Check the messages above for errors.')



There was a problem generating the access token: Bad username or password.  Run the first cell again and enter the credentials.


No access token was generated. Check the messages above for errors.


### Make a query 
After the access token confirmation appears above, you can modify the code below to update the query, then run the cell to save it. In the next cell click run to query for results.
  
Refer to XBRL API documentation at https://xbrlus.github.io/xbrl-api/#/assertion/getAssertionDetails for other endpoints and parameters to filter and return. 

In [2]:
# Define the parameters of the query - this query returns DQC assertions for specified years

endpoint = 'assertion'
XBRL_Elements = [
    'DQC.US.0001.53',
    'DQC.US.0001.56',
    'DQC.US.0001.58',
    'DQC.US.0001.63',
    'DQC.US.0001.71',
    'DQC.US.0001.72',
    'DQC.US.0001.80',
    'DQC.US.0009.39',
    'DQC.US.0009.40',
    'DQC.US.0009.41',
    'DQC.US.0009.42',
    'DQC.US.0009.45',
    'DQC.US.0009.47',
    'DQC.US.0011.6820',
    'DQC.US.0011.6821',
    'DQC.US.0011.6822',
    'DQC.US.0011.6824',
    'DQC.US.0011.6825',
    'DQC.US.0011.6826',
    'DQC.US.0011.6827',
    'DQC.US.0011.6828',
    'DQC.US.0011.6829',
    'DQC.US.0011.6830',
    'DQC.US.0011.6831',
    'DQC.US.0011.6832',
    'DQC.US.0011.9292',
    'DQC.US.0011.9293',
    'DQC.US.0013.2774',
    'DQC.US.0013.2778',
    'DQC.US.0013.2779',
    'DQC.US.0013.2780',
    'DQC.US.0013.2781',
    'DQC.US.0013.2782',
    'DQC.US.0013.2783',
    'DQC.US.0013.2784',
    'DQC.US.0013.2785',
    'DQC.US.0013.2871',
    'DQC.US.0013.2872',
    'DQC.US.0013.2873',
    'DQC.US.0013.2874',
    'DQC.US.0013.3001',
    'DQC.US.0013.3002',
    'DQC.US.0013.3439',
    'DQC.US.0013.3440',
    'DQC.US.0013.3441',
    'DQC.US.0013.9789',
    'DQC.US.0013.9791',
    'DQC.US.0014.2786',
    'DQC.US.0014.2787',
    'DQC.US.0014.2788',
    'DQC.US.0014.2789',
    'DQC.US.0014.2790',
    'DQC.US.0014.2791',
    'DQC.US.0014.2792',
    'DQC.US.0014.2793',
    'DQC.US.0014.2794',
    'DQC.US.0014.2795',
    'DQC.US.0014.2796',
    'DQC.US.0014.2797',
    'DQC.US.0014.2798',
    'DQC.US.0014.2799',
    'DQC.US.0014.2800',
    'DQC.US.0014.2801',
    'DQC.US.0014.2802',
    'DQC.US.0014.2803',
    'DQC.US.0014.2804',
    'DQC.US.0014.2805',
    'DQC.US.0014.2806',
    'DQC.US.0014.2807',
    'DQC.US.0014.2808',
    'DQC.US.0014.2809',
    'DQC.US.0014.3003',
    'DQC.US.0014.3004',
    'DQC.US.0014.3005',
    'DQC.US.0014.3006',
    'DQC.US.0014.3007',
    'DQC.US.0014.3009',
    'DQC.US.0014.3096',
    'DQC.US.0014.7651',
    'DQC.US.0014.7652',
    'DQC.US.0014.9093',
    'DQC.US.0045.6835',
    'DQC.US.0045.6838',
    'DQC.US.0046.6839',
    'DQC.US.0046.6840',
    'DQC.US.0046.6841',
    'DQC.US.0046.6842',
    'DQC.US.0046.7480',
    'DQC.US.0049.7483',
    'DQC.US.0053.7489',
    'DQC.US.0054.7490',
    'DQC.US.0054.7492',
    'DQC.US.0055.9844',
    'DQC.US.0069.7642',
    'DQC.US.0069.7643',
    'DQC.US.0072.7647',
    'DQC.US.0086.9368',
    'DQC.US.0089.9373',
    'DQC.US.0096.9529',
    'DQC.US.0097.9530',
    'DQC.US.0100.9534',
    'DQC.US.0107.9557',
    'DQC.US.0117.10093',
    'DQC.US.0119.9577',
    'DQC.US.0121.9580',
    'DQC.US.0125.9589',
    'DQC.US.0127.9591',
    'DQC.US.0139.9857',
    'DQC.US.0143.9865',
    'DQC.US.0149.9943',
    'DQC.US.0160.10082',
    'DQC.US.0160.10092',
    'DQC.US.0165.10091',
    'DQC.US.0174.10115',
    'DQC.US.0174.10117',
    'DQC.US.0175.10130',
    'DQC.US.0180.10147',
    'DQC.US.0180.10148',
    'DQC.US.0180.10149',
    'DQC.US.0184.10163',
    'DQC.US.0184.10164',
    'DQC.US.0187.10176',
    'DQC.US.0187.10179',
    'DQC.US.0188.10183',
    'DQC.US.0190.10601',
    'DQC.US.0190.10602',
    'DQC.US.0190.10603',
    'DQC.US.0190.10604',
    'DQC.US.0190.10605',
    'DQC.US.0190.10606',
    'DQC.US.0190.10607',
    'DQC.US.0190.10612',
    'DQC.US.0190.10613',
    'DQC.US.0190.10616',
    'DQC.US.0192.10620',
    'DQC.US.0198.10650',
    'DQC.US.0198.10651',
    'DQC.US.0198.10660',
    'DQC.US.0198.10661',
    'DQC.US.0203.10708',
    'DQC.US.0206.10721',
    'DQC.US.0207.10726',
    'DQC.US.0209.10730',
    'DQC.US.0209.10731',
    'DQC.US.0209.10732',
    'DQC.US.0209.10733',
    'DQC.US.0217.10745',
    'DQC.US.0218.10746',
    'DQC.US.0219.10747',
    'DQC.US.0225.10790',
    'DQC.US.0226.10791',
    'DQC.US.0230.10923',
    'DQC.US.0233.10926',
    'DQC.US.0233.10927',
    'DQC.US.0234.10928',
    'DQC.US.0234.10929',
    'DQC.US.0234.10930',
    'DQC.US.0235.10931',
    'DQC.US.0235.10932'
    ]
report_year = [
    'us gaap 2026',
    'us gaap 2025',
    'us gaap 2024'
    ]
fields = [ 
     # this is the list of the characteristics of the data being returned by the query
    'report.base-taxonomy.sort(DESC)',
    'report.filing-date.sort(DESC)',
    'report.entry-url',
    #'report.accepted-timestamp.sort(DESC)',
    'assertion.code.sort(ASC)',
    'report.document-type',
    #'assertion.run-date',
    'report.accession',
    'entity.code',
    'entity.ticker',
    'entity.name',
    'assertion.type',
    #'assertion.detail',
    'assertion.limit()'
    ]

# Set unique rows as True of False (True drops any duplicate rows)
unique = True

# Limit the number of rows displayed by the notebook (does not impact the data frame)
rows_to_display = 2 # Set as '' to display all rows in the notebook

# Below is the list of what's being queried using the search endpoint.
 
params = { 
    'assertion.code': ','.join(XBRL_Elements), 
    'report.base-taxonomy': ','.join(report_year),
    'fields': ','.join(fields)
    }

print('\n\nclick the run button below to execute this query')



click the run button below to execute this query


In [3]:
# @title
# ### Execute the query with loop for all results 
### THIS SECTION DOES NOT NEED TO BE EDITED

search_endpoint = baseurl + 'api/v1/' + endpoint + '/search'
if unique:
    search_endpoint += "?unique"
orig_fields = params['fields']
offset_value = 0
res_df = []
count = 0
query_start = datetime.now()
printed = False
run_query = True

while True:
    if not printed:
        print("On", query_start.strftime("%c"), tokenInfo.email, "(client ID:", str(tokenInfo.client_id.split('-')[0]), "...) started the query and")
        printed = True
    retry = 0
    while retry < 3:
        res = requests.get(search_endpoint, params=params, headers={'Authorization' : 'Bearer {}'.format(tokenInfo.access_token)})
        res_json = res.json()
        if 'error' in res_json:
            if res_json['error_description'] == 'Bad or expired token':
                tokenInfo = refresh(tokenInfo)
            else: 
                print('There was an error: {}'.format(res_json['error_description']))
                run_query = False
                break
        else: 
		        break
        retry +=1
        if retry >= 3:
            print("Can't refresh the access token.  Run the first query block, then rerun the query.")
            run_query = False

    if not run_query:
       break

    print("up to", str(offset_value + res_json['paging']['limit']), "records are found so far ...")

    res_df += res_json['data']

    if res_json['paging']['count'] < res_json['paging']['limit']:
        print(" - this set contained fewer than the", res_json['paging']['limit'], "possible, only", str(res_json['paging']['count']), "records.")
        break
    else: 
        offset_value += res_json['paging']['limit'] 
        if 100 == res_json['paging']['limit']:
                params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                if offset_value == 10 * res_json['paging']['limit']:
                        break 
        elif 500 == res_json['paging']['limit']:
                params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                if offset_value == 4 * res_json['paging']['limit']:
                        break 
        params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)

if not 'error' in res_json:
    current_datetime = datetime.now().replace(microsecond=0)
    time_taken = current_datetime - query_start
    index = pd.DataFrame(res_df).index
    total_rows = len(index)
    your_limit = res_json['paging']['limit']
    limit_message = "If the results below match the limit noted above, you might not be seeing all rows, and should consider upgrading (https://xbrl.us/access-token).\n"
    
    if your_limit == 100:
        print("\nThis non-Member account has a limit of " , 10 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    elif your_limit == 500:
        print("\nThis Basic Individual Member account has a limit of ", 4 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    
    print("\nAt " + current_datetime.strftime("%c") +  ", the query finished with  ", str(total_rows), "  rows returned in " + str(time_taken) + " for \n" +  urllib.parse.unquote(res.url))
    
    df = pd.DataFrame(res_df)
    # the format truncates the HTML display of numerical values to two decimals; .csv data is unaffected
    pd.options.display.float_format = '{:,.2f}'.format
    display(HTML(df.to_html(max_rows=rows_to_display)))

On Fri Apr 24 13:33:19 2026 test.tauriello@xbrl.us (client ID: 69e1257c ...) started the query and
Your access token(None) is refreshed for 60 minutes. If it expires again, run this cell to generate a new token and continue to use the query cells below.
Your access token(None) is refreshed for 60 minutes. If it expires again, run this cell to generate a new token and continue to use the query cells below.
Your access token(None) is refreshed for 60 minutes. If it expires again, run this cell to generate a new token and continue to use the query cells below.
Can't refresh the access token.  Run the first query block, then rerun the query.


The cell below will create a unique list of assertion.codes for the latest taxonomy version

In [4]:
# prompt: In df return only the first of each assertion.code 

df_unique = df.drop_duplicates(subset=['assertion.code'], keep='first')
df_unique


NameError: name 'df' is not defined

The cell below will save the initial dataframe to a local file or Google Drive as a .csv

In [ ]:
# If you run this program locally, you can save the output to a file 
# on your computer (modify D:\results.csv to your system)

df.to_csv(r"D:\assertions-public-exposure.csv",sep=",")

# Google Colab users - comment out the line above and uncomment the code below to save the data frame as a .csv in your Google Drive

#from google.colab import drive
#drive.mount('drive')
#df.to_csv('assertions-public-exposure.csv')
#!cp data.csv "drive/My Drive/"

The code below filters the dataframe by beginning and ending times for SEC filings, then summarizes the dataframe by consolidating assertion code by major rule and adding a column summarizing the number of unique filings for each assertion code. 

In [ ]:
# Convert 'report.accepted-timestamp' to datetime if it's not already
df['report.accepted-timestamp'] = pd.to_datetime(df['report.accepted-timestamp'])

# Define the date range
start_date = '2025-01-01 00:00:00'
end_date = '2025-04-01 00:00:00'

# Filter the dataframe
filtered_df = df[(df['report.accepted-timestamp'] >= start_date) & (df['report.accepted-timestamp'] < end_date)]

# Remove all characters in assertion.code after the third '.' (including the third '.')
filtered_df = filtered_df.map(lambda x: '.'.join(x.split('.')[:3]) if isinstance(x, str) else x)

# Count the occurrences of each assertion.code
assertion_code_counts = filtered_df.groupby('assertion.code').agg(
    count=('assertion.code', 'size'),
    filings=('report.accession', 'nunique')
).reset_index()

# Rename the columns for better readability
assertion_code_counts.columns = ['assertion.code', 'count', 'filings']

# Sort the assertion_code_counts table by 'count' in descending order
assertion_code_counts = assertion_code_counts.sort_values(by='count', ascending=False)

# Display the table
display(HTML(assertion_code_counts.to_html(index=False)))

# Display the updated dataframe
display(HTML(filtered_df.to_html(max_rows=rows_to_display)))

The cell below will report the creation software used for each assertion in the dataframe.

In [ ]:
for accession in df['report.accession'].unique():
  url_report = f"https://api.xbrl.us/api/v1/report/search?report.accession={accession}&fields=report.accession,report.creation-software"
  response_report = requests.get(url_report, headers={'Authorization': 'Bearer {}'.format(tokenInfo.access_token)})
  accession_output = []
  if response_report.status_code == 200:
    report_data = response_report.json()
    #accession_output += report_data['data'][0]
    print(report_data['data'][0])

  elif response_report.status_code == 401:  # Unauthorized, token might have expired
      print("Authorization token expired. Try refreshing it.")
      tokenInfo = refresh(tokenInfo)  # Call your refresh function
  else:
    print(f"Error fetching data for {accession}: Status code {response_report.status_code}, {response_report.text}")

#print(accession_output)